<a href="https://colab.research.google.com/github/EMPEROR82/Automated-MES-Score-Classification-for-Ulcerative-Colitis/blob/main/UGP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install required libraries
!pip install torch torchvision
!pip install scikit-learn
!pip install pyclustering
!pip install torch-geometric
!pip install opencv-python
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 99.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyclustering: filename=pyclustering-0.10.1.2-py3-none-any.whl size=2395100 sha256=b423c1df42b32610b938a5bdf1985c77d6553b2864ac1d843b0be0e79813ef94
  Stored in directory: /root/.cache/pip/wheels/68/29/b4/131bd7deec3663cc311ab9aa64d6517c3e3ec24bcadfc32f74
Successfully built pyclustering
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# /content/drive/MyDrive/UGP/patient_based_classified_images
# /content/drive/MyDrive/UGP/test_set
# /content/drive/MyDrive/UGP/train_and_validation_sets

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score

from pyclustering.cluster.xmeans import xmeans
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer

In [ ]:
class LIMUCDataset(Dataset):

    def __init__(self, img_dir, labels_csv, transform=None):

        self.img_dir = img_dir
        self.labels = pd.read_csv(labels_csv)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        img_path = os.path.join(self.img_dir, self.labels.iloc[idx,0])
        label = int(self.labels.iloc[idx,1])

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_dir = "/content/drive/MyDrive/UGP/train_and_validation_sets"
test_dir = "/content/drive/MyDrive/UGP/test_set"

In [ ]:
# Redefine transform without transforms.ToPILImage()
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# Re-create train_dataset with the corrected transform
train_dataset = ImageFolder(
    root=train_dir,
    transform=transform
)

# Re-create train_loader with the new train_dataset
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

# Original lines from the selected cell
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([0, 2, 2, 0, 3, 0, 0, 3, 3, 0])


In [ ]:
test_dataset = ImageFolder(
    root=test_dir,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

# Optional: Verify the test_loader
images_test, labels_test = next(iter(test_loader))
print("Test images shape:", images_test.shape)
print("Test labels[:10]:", labels_test[:10])

Test images shape: torch.Size([32, 3, 224, 224])
Test labels[:10]: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = models.resnet50(pretrained=True)

# remove final classification layer
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])

resnet = resnet.to(device)
resnet.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 140MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [ ]:
def extract_features(dataloader):

    features = []
    labels = []

    with torch.no_grad():

        for imgs, lbls in tqdm(dataloader):

            imgs = imgs.to(device)

            feat = resnet(imgs)
            feat = feat.view(feat.size(0), -1)

            features.append(feat.cpu().numpy())
            labels.extend(lbls.numpy())

    features = np.concatenate(features, axis=0)
    labels = np.array(labels)

    return features, labels

In [ ]:
train_features, train_labels = extract_features(train_loader)
test_features, test_labels = extract_features(test_loader)

print(train_features.shape)
print(test_features.shape)

100%|██████████| 53/53 [04:38<00:00,  5.26s/it]

(9590, 2048)
(1686, 2048)


In [ ]:
print(train_features.shape)

(9590, 2048)


In [ ]:
import numpy as np

save_path = "/content/drive/MyDrive/UGP/features"

import os
os.makedirs(save_path, exist_ok=True)

np.save(f"{save_path}/train_features.npy", train_features)
np.save(f"{save_path}/train_labels.npy", train_labels)

np.save(f"{save_path}/test_features.npy", test_features)
np.save(f"{save_path}/test_labels.npy", test_labels)

print("Saved successfully.")

Saved successfully.


In [ ]:
!ls -lh /content/drive/MyDrive/UGP/features

total 89M
-rw------- 1 root root 14M Mar 13 10:19 test_features.npy
-rw------- 1 root root 14K Mar 13 10:19 test_labels.npy
-rw------- 1 root root 75M Mar 13 10:19 train_features.npy
-rw------- 1 root root 76K Mar 13 10:19 train_labels.npy


In [ ]:
# features are saved user from here

train_features = np.load("/content/drive/MyDrive/UGP/features/train_features.npy")
train_labels = np.load("/content/drive/MyDrive/UGP/features/train_labels.npy")

test_features = np.load("/content/drive/MyDrive/UGP/features/test_features.npy")
test_labels = np.load("/content/drive/MyDrive/UGP/features/test_labels.npy")

print(train_features.shape)
print(test_features.shape)

(9590, 2048)
(1686, 2048)


In [ ]:
from pyclustering.cluster.xmeans import xmeans
from pyclustering.cluster.center_initializer import kmeans_plusplus_initializer
import warnings
import numpy as np

# Temporarily patch numpy for pyclustering compatibility
# pyclustering might be trying to access numpy.warnings, which is deprecated in newer numpy versions.
# By assigning the standard warnings module to np.warnings, we provide the expected attribute.
if not hasattr(np, 'warnings'):
    np.warnings = warnings

# parameters
initial_k = 2
max_k = 20

# initialize centers
initial_centers = kmeans_plusplus_initializer(
    train_features, initial_k
).initialize()

# run xmeans
xmeans_instance = xmeans(
    train_features,
    initial_centers,
    kmax=max_k
)

xmeans_instance.process()

In [ ]:
clusters = xmeans_instance.get_clusters()
centers = np.array(xmeans_instance.get_centers())

print("Number of clusters discovered:", len(clusters))

Number of clusters discovered: 20


In [ ]:
cluster_labels = np.zeros(len(train_features), dtype=int)

for cluster_id, cluster in enumerate(clusters):
    for index in cluster:
        cluster_labels[index] = cluster_id

In [ ]:
from collections import Counter

cluster_to_class = {}

for cluster_id in range(len(clusters)):

    indices = np.where(cluster_labels == cluster_id)[0]

    labels = train_labels[indices]

    majority_label = Counter(labels).most_common(1)[0][0]

    cluster_to_class[cluster_id] = majority_label

print(cluster_to_class)

{0: np.int64(1), 1: np.int64(0), 2: np.int64(0), 3: np.int64(3), 4: np.int64(0), 5: np.int64(0), 6: np.int64(0), 7: np.int64(0), 8: np.int64(0), 9: np.int64(0), 10: np.int64(0), 11: np.int64(0), 12: np.int64(0), 13: np.int64(0), 14: np.int64(0), 15: np.int64(0), 16: np.int64(1), 17: np.int64(0), 18: np.int64(0), 19: np.int64(0)}


In [ ]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(test_features, centers)

test_cluster_ids = np.argmin(distances, axis=1)

In [ ]:
predictions = np.array([
    cluster_to_class[c] for c in test_cluster_ids
])

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

accuracy = accuracy_score(test_labels, predictions)

f1 = f1_score(
    test_labels,
    predictions,
    average='macro'
)

qwk = cohen_kappa_score(
    test_labels,
    predictions,
    weights='quadratic'
)

print("Accuracy:", accuracy)
print("Macro F1:", f1)
print("Quadratic Weighted Kappa:", qwk)

Accuracy: 0.5551601423487544
Macro F1: 0.3321517163147735
Quadratic Weighted Kappa: 0.3761867579767886


In [ ]:
!pip install torch-geometric

In [ ]:
import torch
from torch_geometric.data import Data

def build_patch_graph(img_tensor, grid_size=7):

    C,H,W = img_tensor.shape
    patch_h = H // grid_size
    patch_w = W // grid_size

    nodes = []

    for i in range(grid_size):
        for j in range(grid_size):

            patch = img_tensor[:,
                               i*patch_h:(i+1)*patch_h,
                               j*patch_w:(j+1)*patch_w]

            nodes.append(patch.mean(dim=(1,2)))

    x = torch.stack(nodes)

    edges = []

    for i in range(grid_size):
        for j in range(grid_size):

            node = i*grid_size + j

            if i < grid_size-1:
                edges.append([node, node+grid_size])
                edges.append([node+grid_size, node])

            if j < grid_size-1:
                edges.append([node, node+1])
                edges.append([node+1, node])

    edge_index = torch.tensor(edges).t().contiguous()

    return Data(x=x, edge_index=edge_index)

In [ ]:
!pip install scikit-image

In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool

class UC_GCN(torch.nn.Module):

    def __init__(self, in_channels):

        super().__init__()

        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64,128)
        self.conv3 = GCNConv(128,256)

        self.fc = torch.nn.Linear(256,4)

    def forward(self, data):

        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index).relu()

        x = global_mean_pool(x, batch)

        return self.fc(x)

In [ ]:
model = UC_GCN(in_channels=3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

criterion = torch.nn.CrossEntropyLoss()

In [ ]:
import torch
from torch_geometric.data import Batch # Import Batch for collating graph data

for epoch in range(20):

    model.train()

    for imgs, lbls in train_loader: # Unpack data into image tensors and label tensors

        # Convert each image in the batch to a torch_geometric Data object
        graph_data_list = []
        for i in range(imgs.shape[0]):
            single_img_tensor = imgs[i] # Get a single image tensor (C, H, W)
            single_label = lbls[i].item() # Get the corresponding label as a Python int

            # Create a Data object for the single image
            graph = build_patch_graph(single_img_tensor) # build_patch_graph expects C,H,W tensor
            graph.y = torch.tensor(single_label, dtype=torch.long) # Assign label to graph

            graph_data_list.append(graph)

        # Collate list of Data objects into a single Batch object
        batched_graph_data = Batch.from_data_list(graph_data_list)

        # Move the entire Batch object (containing all graph tensors) to the device
        batched_graph_data = batched_graph_data.to(device)

        optimizer.zero_grad()

        # Pass the Batch object to the model
        out = model(batched_graph_data)

        # Use the labels from the Batch object for loss calculation
        loss = criterion(out, batched_graph_data.y)

        loss.backward()

        optimizer.step()

In [ ]:
# ===============================
# Install dependencies
# ===============================
!pip install torch-geometric scikit-image -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
from tqdm import tqdm
from skimage.segmentation import slic

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================
# Graph Construction Functions
# ======================================

def build_patch_graph(img_tensor, grid=7):

    C,H,W = img_tensor.shape
    ph = H//grid
    pw = W//grid

    nodes=[]

    for i in range(grid):
        for j in range(grid):
            patch = img_tensor[:,i*ph:(i+1)*ph,j*pw:(j+1)*pw]
            nodes.append(patch.mean(dim=(1,2)))

    x=torch.stack(nodes)

    edges=[]

    for i in range(grid):
        for j in range(grid):

            node=i*grid+j

            if i<grid-1:
                edges.append([node,node+grid])
                edges.append([node+grid,node])

            if j<grid-1:
                edges.append([node,node+1])
                edges.append([node+1,node])

    edge_index=torch.tensor(edges).t().contiguous()

    return x,edge_index


def build_superpixel_graph(img_tensor,n_segments=100):

    img = img_tensor.permute(1,2,0).numpy()

    segments = slic(img,n_segments=n_segments,compactness=10,start_label=0)

    num_nodes=segments.max()+1

    node_features=[]

    for i in range(num_nodes):

        mask=segments==i
        pixels=img[mask]

        node_features.append(pixels.mean(axis=0))

    x=torch.tensor(node_features,dtype=torch.float)

    edges=set()

    H,W=segments.shape

    for i in range(H-1):
        for j in range(W-1):

            a=segments[i,j]
            b=segments[i+1,j]
            c=segments[i,j+1]

            if a!=b:
                edges.add((a,b))
                edges.add((b,a))

            if a!=c:
                edges.add((a,c))
                edges.add((c,a))

    edge_index=torch.tensor(list(edges)).t().contiguous()

    return x,edge_index


# ======================================
# Graph Dataset Builder
# ======================================

def create_graph_dataset(dataset,graph_type="patch"):

    graphs=[]

    for img,label in tqdm(dataset):

        img=img.cpu()

        if graph_type=="patch":
            x,edge_index = build_patch_graph(img)

        else:
            x,edge_index = build_superpixel_graph(img)

        graphs.append(
            Data(
                x=x.float(),
                edge_index=edge_index,
                y=torch.tensor(label)
            )
        )

    return graphs


# ======================================
# GCN Model
# ======================================

class UC_GCN(nn.Module):

    def __init__(self,in_channels):

        super().__init__()

        self.conv1=GCNConv(in_channels,64)
        self.conv2=GCNConv(64,128)
        self.conv3=GCNConv(128,256)

        self.fc=nn.Linear(256,4)

    def forward(self,data):

        x,edge_index,batch=data.x,data.edge_index,data.batch

        x=self.conv1(x,edge_index)
        x=F.relu(x)

        x=self.conv2(x,edge_index)
        x=F.relu(x)

        x=self.conv3(x,edge_index)
        x=F.relu(x)

        x=global_mean_pool(x,batch)

        return self.fc(x)


# ======================================
# Training Function
# ======================================

def train_model(train_loader,test_loader,in_channels,epochs=15):

    model=UC_GCN(in_channels).to(device)

    optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

    criterion=nn.CrossEntropyLoss()

    for epoch in range(epochs):

        model.train()
        losses=[]

        for data in train_loader:

            data=data.to(device)

            optimizer.zero_grad()

            out=model(data)

            loss=criterion(out,data.y)

            loss.backward()

            optimizer.step()

            losses.append(loss.item())

        print(f"Epoch {epoch+1} Loss:",np.mean(losses))

    # evaluation
    model.eval()

    preds=[]
    trues=[]

    with torch.no_grad():

        for data in test_loader:

            data=data.to(device)

            out=model(data)

            pred=out.argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            trues.extend(data.y.cpu().numpy())

    acc=accuracy_score(trues,preds)
    f1=f1_score(trues,preds,average="macro")
    qwk=cohen_kappa_score(trues,preds,weights="quadratic")

    print("Accuracy:",acc)
    print("F1:",f1)
    print("QWK:",qwk)

    return model


# ======================================
# Build Patch Graph Dataset
# ======================================

print("Building Patch Graphs...")

patch_train = create_graph_dataset(train_dataset,"patch")
patch_test = create_graph_dataset(test_dataset,"patch")

patch_train_loader = GeoLoader(patch_train,batch_size=32,shuffle=True)
patch_test_loader = GeoLoader(patch_test,batch_size=32)

# ======================================
# Train Patch GCN
# ======================================

print("\nTraining Patch GCN")

patch_model = train_model(
    patch_train_loader,
    patch_test_loader,
    in_channels=3
)


# ======================================
# Build Superpixel Graph Dataset
# ======================================

print("\nBuilding Superpixel Graphs...")

sp_train = create_graph_dataset(train_dataset,"superpixel")
sp_test = create_graph_dataset(test_dataset,"superpixel")

sp_train_loader = GeoLoader(sp_train,batch_size=16,shuffle=True)
sp_test_loader = GeoLoader(sp_test,batch_size=16)

# ======================================
# Train Superpixel GCN
# ======================================

print("\nTraining Superpixel GCN")

sp_model = train_model(
    sp_train_loader,
    sp_test_loader,
    in_channels=3
)

Building Patch Graphs...


100%|██████████| 1686/1686 [00:16<00:00, 100.67it/s]



Training Patch GCN
Epoch 1 Loss: 1.056371269027392
Epoch 2 Loss: 0.9904399156570435
Epoch 3 Loss: 0.9819592940807342
Epoch 4 Loss: 0.9790324930349986
Epoch 5 Loss: 0.9791010349988938
Epoch 6 Loss: 0.9766838272412618
Epoch 7 Loss: 0.9784482735395431
Epoch 8 Loss: 0.9778462292750677
Epoch 9 Loss: 0.9789727979898453
Epoch 10 Loss: 0.975886244972547
Epoch 11 Loss: 0.9757640681664149
Epoch 12 Loss: 0.9731836668650309
Epoch 13 Loss: 0.9737087164322535
Epoch 14 Loss: 0.9746315636237463
Epoch 15 Loss: 0.9729007059335708
Accuracy: 0.5771055753262159
F1: 0.3043379246611512
QWK: 0.2901976708242108

Building Superpixel Graphs...


  0%|          | 0/9590 [00:00<?, ?it/s]/tmp/ipykernel_1741/314031020.py:79: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  x=torch.tensor(node_features,dtype=torch.float)
100%|██████████| 1686/1686 [03:08<00:00,  8.97it/s]



Training Superpixel GCN
Epoch 1 Loss: 1.0414117815593877
Epoch 2 Loss: 0.9862447781364123
Epoch 3 Loss: 0.9802138942976792
Epoch 4 Loss: 0.9790086585779985
Epoch 5 Loss: 0.9817244099080562
Epoch 6 Loss: 0.9765328515569369
Epoch 7 Loss: 0.9767771936456362
Epoch 8 Loss: 0.9737022809684277
Epoch 9 Loss: 0.972303822139899
Epoch 10 Loss: 0.9720240421096484
Epoch 11 Loss: 0.9713572105268637
Epoch 12 Loss: 0.9702784970402718
Epoch 13 Loss: 0.969632975757122
Epoch 14 Loss: 0.9695508232712746
Epoch 15 Loss: 0.9680732102195422
Accuracy: 0.5990510083036773
F1: 0.3111209174108845
QWK: 0.25183970222678576


In [ ]:
import torch
from torch_geometric.data import Data

def build_patch_graph_improved(img_tensor, grid_size=7):

    C,H,W = img_tensor.shape
    patch_h = H // grid_size
    patch_w = W // grid_size

    nodes = []

    for i in range(grid_size):
        for j in range(grid_size):

            patch = img_tensor[:,
                               i*patch_h:(i+1)*patch_h,
                               j*patch_w:(j+1)*patch_w]

            # nodes.append(patch.mean(dim=(1,2)))
            feat = resnet(patch.unsqueeze(0).to(device))
            feat = feat.view(-1).cpu()
            nodes.append(feat)

    x = torch.stack(nodes)

    edges = []

    for i in range(grid_size):
        for j in range(grid_size):

            node = i*grid_size + j

            if i < grid_size-1:
                edges.append([node, node+grid_size])
                edges.append([node+grid_size, node])

            if j < grid_size-1:
                edges.append([node, node+1])
                edges.append([node+1, node])

    edge_index = torch.tensor(edges).t().contiguous()

    return Data(x=x, edge_index=edge_index)

ModuleNotFoundError: No module named 'torch_geometric'